# 🏘️ Pune Real Estate — End-to-End ML Pipeline
**Covers:** Data upload → Cleaning → Feature Engineering → Model Training → Evaluation → MLflow → PyCaret → FastAPI → DVC

---
## ⬆️ HOW TO UPLOAD YOUR FILES TO COLAB
Run the cell below — it opens a file picker dialog right in Colab.

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1: Upload your files to Colab (run this FIRST)
# ─────────────────────────────────────────────────────────────
from google.colab import files
import os

os.makedirs('/content/data/raw', exist_ok=True)
os.makedirs('/content/data/processed', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)

print('📁 File picker will open — select BOTH files at once:')
print('   1. Pune_Real_Estate_Data__1_.xlsx')
print('   2. data_cleaned__1_.csv')
print()
uploaded = files.upload()

# Move to data/raw/
for fname in uploaded:
    dest = f'/content/data/raw/{fname}'
    os.rename(fname, dest)
    print(f'✅  Saved → {dest}')

print('\nFiles in /content/data/raw/:')
print(os.listdir('/content/data/raw/'))

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2: Install dependencies
# ─────────────────────────────────────────────────────────────
!pip install -q mlflow pycaret[full] fastapi uvicorn joblib openpyxl

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3: Imports
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, joblib
warnings.filterwarnings('ignore')

# Detect xlsx/csv filenames dynamically
raw_files = os.listdir('/content/data/raw/')
XLSX_FILE = next((f for f in raw_files if f.endswith('.xlsx')), None)
CSV_FILE  = next((f for f in raw_files if f.endswith('.csv')),  None)

print(f'XLSX: {XLSX_FILE}')
print(f'CSV : {CSV_FILE}')

## 📊 Step 1: Exploratory Data Analysis

In [ ]:
# Load raw data
df_raw  = pd.read_excel(f'/content/data/raw/{XLSX_FILE}')
df_base = pd.read_csv(f'/content/data/raw/{CSV_FILE}')

print('=== Raw XLSX ===' )
print(df_raw.shape)
df_raw.head(3)

In [ ]:
print('Missing values in XLSX:')
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print()
print('Missing values in CSV:')
print(df_base.isnull().sum()[df_base.isnull().sum() > 0])

In [ ]:
# EDA — Price distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(df_base['Price Cleaned'].dropna(), bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Price Distribution (Lakhs)')
axes[0].set_xlabel('Price (Lakhs ₹)')

axes[1].hist(np.log1p(df_base['Price Cleaned'].dropna()), bins=30, color='tomato', edgecolor='white')
axes[1].set_title('Log-Price Distribution')
axes[1].set_xlabel('log(Price)')

plt.tight_layout()
plt.show()

In [ ]:
# Amenity correlation heatmap
amenity_cols = ['ClubHouse Cleaned','School Cleaned','Hospital Cleaned',
                'Mall Cleaned','Park Cleaned','Pool Cleaned','Gym Cleaned']
corr_cols = amenity_cols + ['Area Cleaned', 'Price Cleaned']
corr = df_base[corr_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 🧹 Step 2: Data Cleaning & Feature Engineering

In [ ]:
def clean_raw(df):
    df = df.copy()
    df.columns = (df.columns.str.strip().str.lower()
                    .str.replace(r'[^\w]+', '_', regex=True))
    df.rename(columns={'propert_type': 'property_type'}, inplace=True)

    df['property_area_sqft'] = pd.to_numeric(
        df['property_area_in_sq_ft'].astype(str).str.replace(',','').str.extract(r'([\d.]+)')[0],
        errors='coerce')
    df['price_lakhs'] = pd.to_numeric(
        df['price_in_lakhs'].astype(str).str.replace(',','').str.extract(r'([\d.]+)')[0],
        errors='coerce')

    binary = {'clubhouse':'has_clubhouse','school___university_in_township_':'has_school',
              'hospital_in_township':'has_hospital','mall_in_township':'has_mall',
              'park___jogging_track':'has_park','swimming_pool':'has_pool','gym':'has_gym'}
    yes_map = {'yes':1,'no':0,'1':1,'0':0}
    for src, tgt in binary.items():
        if src in df.columns:
            df[tgt] = (df[src].astype(str).str.strip().str.lower()
                         .map(yes_map).fillna(0).astype(int))

    for col in ['location','sub_area','property_type','company_name']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower()

    df.dropna(subset=['price_lakhs','property_area_sqft'], inplace=True)
    return df


def engineer_features(df_clean, df_base):
    shared = min(len(df_clean), len(df_base))
    df = df_clean.iloc[:shared].reset_index(drop=True).copy()
    df['area_sqft']   = df_base['Area Cleaned'].iloc[:shared].values
    df['price_lakhs'] = df_base['Price Cleaned'].iloc[:shared].values

    flag_map = {'ClubHouse Cleaned':'has_clubhouse','School Cleaned':'has_school',
                'Hospital Cleaned':'has_hospital','Mall Cleaned':'has_mall',
                'Park Cleaned':'has_park','Pool Cleaned':'has_pool','Gym Cleaned':'has_gym'}
    for src, tgt in flag_map.items():
        if src in df_base.columns:
            df[tgt] = df_base[src].iloc[:shared].values

    df['amenity_score']  = df[[v for v in flag_map.values()]].sum(axis=1)
    df['price_per_sqft'] = df['price_lakhs'] / (df['area_sqft'] + 1e-6)
    df['log_area']       = np.log1p(df['area_sqft'])
    df['log_price']      = np.log1p(df['price_lakhs'])

    if 'total_township_area_in_acres' in df.columns:
        df['township_area'] = df['total_township_area_in_acres'].fillna(
            df['total_township_area_in_acres'].median())
    else:
        df['township_area'] = 0.0

    for col in ['location','sub_area','property_type','company_name',
                'township_name__society_name']:
        if col in df.columns:
            df[col] = df[col].astype('category').cat.codes

    df.dropna(subset=['price_lakhs','area_sqft'], inplace=True)
    return df.reset_index(drop=True)


df_clean    = clean_raw(df_raw)
df_features = engineer_features(df_clean, df_base)
df_features.to_csv('/content/data/processed/pune_features.csv', index=False)
print(f'✅  Features saved — shape: {df_features.shape}')
df_features.head(3)

## 🤖 Step 3: Model Training & Evaluation with MLflow

In [ ]:
import mlflow, mlflow.sklearn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FEATURE_COLS = ['area_sqft','log_area','township_area','amenity_score',
                'has_clubhouse','has_school','has_hospital','has_mall',
                'has_park','has_pool','has_gym',
                'location','sub_area','property_type','company_name']
TARGET = 'price_lakhs'

available = [c for c in FEATURE_COLS if c in df_features.columns]
X = df_features[available]
y = df_features[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')),
                       ('sc', StandardScaler())]), num_cols)
], remainder='drop')

mlflow.set_tracking_uri('/content/mlruns')
mlflow.set_experiment('pune_re_price_prediction')

models = {
    'Ridge':        Ridge(alpha=10),
    'Lasso':        Lasso(alpha=1),
    'RandomForest': RandomForestRegressor(n_estimators=200, random_state=42),
    'GBM':          GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42),
    'ExtraTrees':   ExtraTreesRegressor(n_estimators=200, random_state=42),
}

results = {}
best_r2, best_name, best_pipe = -np.inf, None, None

for name, est in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', est)])
    with mlflow.start_run(run_name=name):
        pipe.fit(X_train, y_train)
        preds = pipe.predict(X_test)
        r2   = r2_score(y_test, preds)
        mae  = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        cv   = cross_val_score(pipe, X, y, cv=5, scoring='r2').mean()
        mlflow.log_params({'model': name})
        mlflow.log_metrics({'r2': r2, 'mae': mae, 'rmse': rmse, 'cv_r2': cv})
        mlflow.sklearn.log_model(pipe, 'model')
        results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse, 'CV_R2': cv}
        flag = ' ← BEST' if r2 > best_r2 else ''
        if r2 > best_r2:
            best_r2, best_name, best_pipe = r2, name, pipe
        print(f'{name:15s}  R²={r2:.3f}  MAE={mae:.1f}L  RMSE={rmse:.1f}L  CV={cv:.3f}{flag}')

# Save best model
joblib.dump({'pipeline': best_pipe, 'features': available}, '/content/models/best_model.pkl')
print(f'\n✅  Best: {best_name}  R²={best_r2:.3f}')

In [ ]:
# Results table
pd.DataFrame(results).T.sort_values('R2', ascending=False).round(3)

In [ ]:
# Feature importance (if RF or GBM)
if hasattr(best_pipe.named_steps['model'], 'feature_importances_'):
    imp = best_pipe.named_steps['model'].feature_importances_
    feat_names = best_pipe.named_steps['prep'].transformers_[0][2]
    fi = pd.Series(imp, index=feat_names).sort_values(ascending=False)
    fi.plot(kind='bar', figsize=(10, 4), title=f'{best_name} — Feature Importances', color='steelblue')
    plt.tight_layout()
    plt.show()

## 🚀 Step 4: PyCaret AutoML

In [ ]:
from pycaret.regression import setup, compare_models, tune_model, finalize_model, save_model, pull

df_pc = df_features[available + [TARGET]].dropna(subset=[TARGET])
print(f'PyCaret input shape: {df_pc.shape}')

exp = setup(data=df_pc, target=TARGET, session_id=42, train_size=0.8,
            normalize=True, transformation=True, fold=5, verbose=False, html=False)

best_models = compare_models(n_select=3, sort='R2', verbose=True)
lb = pull()
lb[['Model','R2','MAE','RMSE']].head(5)

In [ ]:
# Tune best model
best_pc = best_models[0] if isinstance(best_models, list) else best_models
tuned_pc = tune_model(best_pc, optimize='R2', n_iter=20)
final_pc = finalize_model(tuned_pc)
save_model(final_pc, '/content/models/pycaret_best')
print('✅  PyCaret model saved')

## 🌐 Step 5: FastAPI — Test In-Notebook

In [ ]:
# Write FastAPI app
fastapi_code = '''
import os, joblib, numpy as np, pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

bundle   = joblib.load("/content/models/best_model.pkl")
pipeline = bundle["pipeline"]
features = bundle["features"]

app = FastAPI(title="Pune RE Price API")

class Prop(BaseModel):
    area_sqft: float = 900
    township_area: float = 50
    amenity_score: int = 3
    has_clubhouse: int = 1
    has_school: int = 1
    has_hospital: int = 0
    has_mall: int = 0
    has_park: int = 1
    has_pool: int = 1
    has_gym: int = 1
    location: int = 3
    sub_area: int = 5
    property_type: int = 1
    company_name: int = 2

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/predict")
def predict(p: Prop):
    data = {"area_sqft":p.area_sqft,"log_area":np.log1p(p.area_sqft),
            "township_area":p.township_area,"amenity_score":p.amenity_score,
            "has_clubhouse":p.has_clubhouse,"has_school":p.has_school,
            "has_hospital":p.has_hospital,"has_mall":p.has_mall,
            "has_park":p.has_park,"has_pool":p.has_pool,"has_gym":p.has_gym,
            "location":p.location,"sub_area":p.sub_area,
            "property_type":p.property_type,"company_name":p.company_name}
    df_in = pd.DataFrame([{k:data[k] for k in features if k in data}])
    pred  = float(pipeline.predict(df_in)[0])
    return {"predicted_price_lakhs": round(pred, 2),
            "predicted_price_millions": round(pred/10, 3)}
'''
with open('/content/api_app.py', 'w') as f:
    f.write(fastapi_code)
print('FastAPI app written.')

In [ ]:
# Start FastAPI in background using pyngrok tunnel
!pip install -q pyngrok

import subprocess, time
from pyngrok import ngrok

# Start uvicorn
proc = subprocess.Popen(['uvicorn', 'api_app:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(3)

# Open ngrok tunnel
public_url = ngrok.connect(8000)
print(f'\n🌐 Public API URL: {public_url}')
print(f'   Swagger docs : {public_url}/docs')
print(f'   Health check : {public_url}/health')

In [ ]:
# Test the API
import requests

payload = {
    'area_sqft': 1000, 'township_area': 80, 'amenity_score': 5,
    'has_clubhouse': 1, 'has_school': 1, 'has_hospital': 1,
    'has_mall': 0, 'has_park': 1, 'has_pool': 1, 'has_gym': 1,
    'location': 3, 'sub_area': 5, 'property_type': 1, 'company_name': 2
}

r = requests.post(f'{public_url}/predict', json=payload)
print('Status:', r.status_code)
print('Response:', r.json())

## 📦 Step 6: Download all outputs

In [ ]:
# Download processed data and models
from google.colab import files

for path in [
    '/content/data/processed/pune_features.csv',
    '/content/models/best_model.pkl',
]:
    if os.path.exists(path):
        files.download(path)
        print(f'Downloaded: {path}')